# Custom Material Composition: U-Zr and U-Pu-Zr Alloys

This notebook explores how **material composition** affects gas swelling behavior in nuclear fuel. You'll learn how to customize parameters for different alloy compositions and understand the physical origins of these differences.

## Learning Objectives

By the end of this tutorial, you will:
- Understand how alloy composition affects swelling behavior
- Learn key parameter differences between U-Zr and U-Pu-Zr alloys
- Explore Zr content variations in U-Zr alloys
- Investigate Pu content effects in U-Pu-Zr alloys
- Create custom material parameter sets for your research
- Compare swelling behavior across different compositions

## Why Material Composition Matters

Different fuel compositions have different swelling characteristics due to:

1. **Crystal Structure Variations**: Alloying elements change lattice parameters
2. **Defect Dynamics**: Dislocation density and mobility vary with composition
3. **Phase Behavior**: Different phases form at various compositions
4. **Gas Bubble Nucleation**: Boundary characteristics affect bubble formation

This notebook helps you choose the right parameters for your specific fuel composition.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from gas_swelling.models.modelrk23 import GasSwellingModel
from gas_swelling.params.parameters import MaterialParameters, SimulationParameters, create_default_parameters

# Configure plotting for better visuals
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Libraries imported successfully!")
print("\nReady to explore material composition effects.")

## 1. Overview of Standard Fuel Compositions

The gas swelling model has been validated against experimental data for three main fuel types. Let's examine their key parameter differences.

In [ ]:
# Define standard material parameter sets for different fuel types

def create_u10zr_parameters():
    """U-10% Zr alloy (most common fast reactor fuel)"""
    material = MaterialParameters(
        dislocation_density=7.0e13,  # m^-2 - moderate dislocation density
        Fnb=1e-5,                     # Bulk nucleation factor
        Fnf=1e-5,                     # Boundary nucleation factor
        surface_energy=0.5            # J/m^2
    )
    sim = SimulationParameters(
        temperature=700,              # K - peak swelling temperature for U-10Zr
        fission_rate=2e20,            # fissions/m^3/s
        max_time=3600*24*100          # 100 days
    )
    return material, sim

def create_u19pu10zr_parameters():
    """U-19% Pu-10% Zr alloy (advanced burner reactor fuel)"""
    material = MaterialParameters(
        dislocation_density=2.0e13,  # m^-2 - lower than U-10Zr (reduced swelling)
        Fnb=1e-5,                     # Same bulk nucleation
        Fnf=1e-5,                     # Same boundary nucleation
        surface_energy=0.5            # J/m^2
    )
    sim = SimulationParameters(
        temperature=750,              # K - higher peak temperature for U-Pu-Zr
        fission_rate=2e20,
        max_time=3600*24*100
    )
    return material, sim

def create_pure_uranium_parameters():
    """High-purity uranium (research/historical studies only)"""
    material = MaterialParameters(
        dislocation_density=1.0e15,  # m^-2 - very high (cold-worked)
        Fnb=1e-5,
        Fnf=1.0,                      # 5 orders of magnitude higher!
        surface_energy=0.5
    )
    sim = SimulationParameters(
        temperature=673,              # K - lower peak temperature
        fission_rate=2e20,
        max_time=3600*24*100
    )
    return material, sim

# Display comparison table
print("=" * 80)
print("STANDARD FUEL COMPOSITION PARAMETER COMPARISON")
print("=" * 80)

fuel_types = [
    ("U-10Zr", create_u10zr_parameters),
    ("U-19Pu-10Zr", create_u19pu10zr_parameters),
    ("High-Purity U", create_pure_uranium_parameters)
]

print(f"{'Fuel Type':<20} {'Dislocation (m^-2)':<20} {'Fnf':<15} {'Peak T (K)':<12}")
print("-" * 80)

for name, func in fuel_types:
    mat, sim = func()
    print(f"{name:<20} {mat.dislocation_density:.2e}        {mat.Fnf:<15.2e} {sim.temperature:<12.0f}")

print("\n" + "=" * 80)
print("KEY OBSERVATIONS:")
print("=" * 80)
print("• U-Pu-Zr has LOWER dislocation density → lower swelling")
print("• High-purity U has VERY HIGH Fnf (1.0) → extreme swelling behavior")
print("• Peak swelling temperature varies: 673K (pure U) → 750K (U-Pu-Zr)")
print("=" * 80)

## 2. Comparing Swelling Behavior of Standard Compositions

Let's run simulations for all three fuel types and compare their swelling evolution.

In [ ]:
# Simulation time: 100 days
sim_time_sec = 3600 * 24 * 100
t_eval = np.linspace(0, sim_time_sec, 100)
time_days = t_eval / (24 * 3600)

print("Running simulations for standard fuel compositions...")
print("(This may take 30-60 seconds...)\n")

# Store results
results = {}

for name, func in fuel_types:
    print(f"Simulating {name}...", end=" ")
    
    # Create parameters
    material, sim = func()
    params = create_default_parameters()
    
    # Update with material-specific values
    for key, value in material.__dict__.items():
        if key in params:
            params[key] = value
    for key, value in sim.__dict__.items():
        if key in params:
            params[key] = value
    
    # Recalculate diffusion coefficients at material's temperature
    from gas_swelling.params.parameters import BOLTZMANN_CONSTANT_EV
    Dgb = (sim.Dgb_prefactor * np.exp(-sim.Dgb_activation_energy / (BOLTZMANN_CONSTANT_EV * sim.temperature)) + 
          sim.Dgb_fission_term * sim.fission_rate)
    params['Dgb'] = Dgb
    params['Dgf'] = Dgb * sim.Dgf_multiplier
    params['temperature'] = sim.temperature
    
    # Run simulation
    model = GasSwellingModel(params)
    result = model.solve(t_span=(0, sim_time_sec), t_eval=t_eval)
    
    # Calculate swelling
    Vb = (4/3) * np.pi * result['Rcb']**3 * result['Ccb']
    Vf = (4/3) * np.pi * result['Rcf']**3 * result['Ccf']
    swelling = (Vb + Vf) * 100
    
    results[name] = {
        'swelling': swelling,
        'Rcb': result['Rcb'],
        'Rcf': result['Rcf'],
        'Ccb': result['Ccb'],
        'Ccf': result['Ccf'],
        'temperature': sim.temperature
    }
    
    print(f"✓ Final swelling: {swelling[-1]:.4f}%")

print("\n✅ All simulations completed!")

In [ ]:
# Plot comparison of swelling evolution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Standard Fuel Composition Comparison', fontsize=14, fontweight='bold')

# Plot 1: Swelling evolution
colors = {'U-10Zr': 'blue', 'U-19Pu-10Zr': 'green', 'High-Purity U': 'red'}
for name in results:
    axes[0, 0].plot(time_days, results[name]['swelling'], 
                   linewidth=2, color=colors[name], label=name)
axes[0, 0].set_xlabel('Time (days)', fontsize=11)
axes[0, 0].set_ylabel('Swelling (%)', fontsize=11)
axes[0, 0].set_title('Swelling Evolution', fontweight='bold')
axes[0, 0].legend()

# Plot 2: Final swelling comparison
fuel_names = list(results.keys())
final_swelling = [results[name]['swelling'][-1] for name in fuel_names]
bars = axes[0, 1].bar(range(len(fuel_names)), final_swelling, 
                       color=[colors[name] for name in fuel_names])
axes[0, 1].set_xticks(range(len(fuel_names)))
axes[0, 1].set_xticklabels(fuel_names, rotation=15, ha='right')
axes[0, 1].set_ylabel('Final Swelling (%)', fontsize=11)
axes[0, 1].set_title('Swelling at 100 Days', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}%', ha='center', va='bottom', fontsize=9)

# Plot 3: Bubble radius comparison
for name in results:
    axes[1, 0].plot(time_days, results[name]['Rcb']*1e9,
                   linewidth=2, color=colors[name], label=f'{name} (bulk)')
axes[1, 0].set_xlabel('Time (days)', fontsize=11)
axes[1, 0].set_ylabel('Bulk Bubble Radius (nm)', fontsize=11)
axes[1, 0].set_title('Bubble Growth', fontweight='bold')
axes[1, 0].legend()

# Plot 4: Bubble concentration comparison
for name in results:
    axes[1, 1].semilogy(time_days, results[name]['Ccb'],
                       linewidth=2, color=colors[name], label=name)
axes[1, 1].set_xlabel('Time (days)', fontsize=11)
axes[1, 1].set_ylabel('Bubble Concentration (m⁻³)', fontsize=11)
axes[1, 1].set_title('Bubble Nucleation', fontweight='bold')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("\n🔍 PHYSICAL INTERPRETATION:")
print("="*60)
for name in fuel_names:
    temp = results[name]['temperature']
    swell = results[name]['swelling'][-1]
    print(f"{name}:")
    print(f"  Temperature: {temp} K")
    print(f"  Final swelling: {swell:.4f}%")
    if name == 'U-19Pu-10Zr':
        print(f"  → {final_swelling[0]/swell*100-100:.1f}% less swelling than U-10Zr")
    if name == 'High-Purity U':
        print(f"  → {swell/final_swelling[0]:.1f}x more swelling than U-10Zr")
    print()

## 3. U-Zr Alloy Variations: Effect of Zirconium Content

Zirconium content affects swelling behavior primarily through changes in dislocation density and phase stability. Let's explore different Zr contents.

In [ ]:
# Define Zr content variations to study
zr_contents = [6, 8, 10, 12]  # weight percent Zr

# Empirical relationship: dislocation density varies with Zr content
# Based on: higher Zr → more phase boundaries → moderate dislocation density
def estimate_dislocation_density_zr(zr_content):
    """
    Estimate dislocation density based on Zr content.
    
    Physical basis:
    - Lower Zr (6%): More U-rich phase → higher dislocation density
    - Medium Zr (10%): Balanced phases → moderate dislocation density
    - Higher Zr (12%): More Zr-rich phase → lower dislocation density
    """
    # Base dislocation density at 10% Zr
    rho_base = 7.0e13  # m^-2
    
    # Linear variation: ±30% for ±4% Zr from 10%
    factor = 1.0 + 0.075 * (10 - zr_content)
    
    return rho_base * factor

# Peak swelling temperature also varies with Zr content
def estimate_peak_temperature_zr(zr_content):
    """
    Estimate peak swelling temperature based on Zr content.
    
    Higher Zr content raises the peak temperature due to:
    - Increased phase stability
    - Modified diffusion characteristics
    """
    # Base: 700 K at 10% Zr
    # Increase by ~5 K per additional % Zr
    return 700 + 5 * (zr_content - 10)

# Display parameter estimates
print("=" * 80)
print("U-ZR ALLOY: PARAMETER VARIATION WITH ZR CONTENT")
print("=" * 80)
print(f"{'Zr Content (wt%)':<20} {'Dislocation (m^-2)':<25} {'Peak T (K)':<15}")
print("-" * 80)

zr_params = {}
for zr in zr_contents:
    rho = estimate_dislocation_density_zr(zr)
    T_peak = estimate_peak_temperature_zr(zr)
    zr_params[zr] = {'rho': rho, 'T_peak': T_peak}
    print(f"U-{zr}%Zr{'':<12} {rho:.2e}            {T_peak:<15.0f}")

print("\n" + "=" * 80)
print("PHYSICAL INTERPRETATION:")
print("=" * 80)
print("• Lower Zr content → higher dislocation density → potentially higher swelling")
print("• Higher Zr content → higher peak temperature → shifts swelling to higher T")
print("• Optimal swelling range: 8-10% Zr for fast reactor applications")
print("=" * 80)

In [ ]:
# Run simulations for different Zr contents
print("Running simulations for U-Zr alloys with varying Zr content...")
print("(This may take 30-60 seconds...)\n")

zr_results = {}

for zr in zr_contents:
    print(f"Simulating U-{zr}%Zr...", end=" ")
    
    # Get estimated parameters
    rho = zr_params[zr]['rho']
    T = zr_params[zr]['T_peak']
    
    # Create parameter set
    params = create_default_parameters()
    params['dislocation_density'] = rho
    params['temperature'] = T
    params['fission_rate'] = 2e20
    
    # Recalculate diffusion at this temperature
    from gas_swelling.params.parameters import BOLTZMANN_CONSTANT_EV
    Dgb = (params['Dgb_prefactor'] * np.exp(-params['Dgb_activation_energy'] / (BOLTZMANN_CONSTANT_EV * T)) + 
          params['Dgb_fission_term'] * params['fission_rate'])
    params['Dgb'] = Dgb
    params['Dgf'] = Dgb * params['Dgf_multiplier']
    
    # Run simulation
    model = GasSwellingModel(params)
    result = model.solve(t_span=(0, sim_time_sec), t_eval=t_eval)
    
    # Calculate swelling
    Vb = (4/3) * np.pi * result['Rcb']**3 * result['Ccb']
    Vf = (4/3) * np.pi * result['Rcf']**3 * result['Ccf']
    swelling = (Vb + Vf) * 100
    
    zr_results[zr] = {
        'swelling': swelling,
        'Rcb': result['Rcb'],
        'Ccb': result['Ccb'],
        'T': T
    }
    
    print(f"✓ T={T}K, Swelling={swelling[-1]:.4f}%")

print("\n✅ All Zr content simulations completed!")

In [ ]:
# Visualize Zr content effects
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Effect of Zirconium Content in U-Zr Alloys', fontsize=14, fontweight='bold')

# Plot 1: Swelling evolution
colors_zr = plt.cm.viridis(np.linspace(0.2, 0.8, len(zr_contents)))
for i, zr in enumerate(zr_contents):
    axes[0].plot(time_days, zr_results[zr]['swelling'],
               linewidth=2, color=colors_zr[i], label=f'U-{zr}%Zr')
axes[0].set_xlabel('Time (days)', fontsize=11)
axes[0].set_ylabel('Swelling (%)', fontsize=11)
axes[0].set_title('Swelling Evolution', fontweight='bold')
axes[0].legend()

# Plot 2: Final swelling vs Zr content
final_swelling_zr = [zr_results[zr]['swelling'][-1] for zr in zr_contents]
axes[1].plot(zr_contents, final_swelling_zr, 'o-', linewidth=2, markersize=10, color='teal')
axes[1].set_xlabel('Zr Content (wt%)', fontsize=11)
axes[1].set_ylabel('Final Swelling (%)', fontsize=11)
axes[1].set_title('Zr Content Effect', fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Add value labels
for i, (zr, swell) in enumerate(zip(zr_contents, final_swelling_zr)):
    axes[1].text(zr, swell, f'{swell:.3f}%', ha='center', va='bottom', fontsize=9)

# Plot 3: Peak temperature vs Zr content
peak_temps = [zr_results[zr]['T'] for zr in zr_contents]
axes[2].plot(zr_contents, peak_temps, 's-', linewidth=2, markersize=10, color='orange')
axes[2].set_xlabel('Zr Content (wt%)', fontsize=11)
axes[2].set_ylabel('Peak Temperature (K)', fontsize=11)
axes[2].set_title('Peak Temperature Shift', fontweight='bold')
axes[2].grid(True, alpha=0.3)

# Add value labels
for i, (zr, T) in enumerate(zip(zr_contents, peak_temps)):
    axes[2].text(zr, T+2, f'{T}K', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\n📊 SUMMARY: ZR CONTENT EFFECTS")
print("="*60)
print("Swelling varies by {:.1f}% across 6-12% Zr range".format(
    (max(final_swelling_zr) - min(final_swelling_zr))/min(final_swelling_zr)*100
))
print("Peak temperature shifts by {}K".format(max(peak_temps) - min(peak_temps)))
print("\nRecommendation: U-10Zr offers good balance of swelling and temperature.")

## 4. U-Pu-Zr Alloy Variations: Effect of Plutonium Content

Plutonium content significantly affects swelling behavior. Higher Pu content generally reduces swelling due to changes in defect dynamics.

In [ ]:
# Define Pu content variations (keeping Zr constant at 10%)
pu_contents = [10, 15, 19, 25]  # weight percent Pu (typical range: 10-30%)

# Empirical relationships for U-Pu-Zr alloys
def estimate_dislocation_density_pu(pu_content, zr_content=10):
    """
    Estimate dislocation density based on Pu content.
    
    Physical basis:
    - Higher Pu → more alloying → lower dislocation density
    - Pu affects phase distribution and defect mobility
    """
    # Base: U-10Zr without Pu
    rho_base = 7.0e13  # m^-2
    
    # Each % Pu reduces dislocation density by ~4%
    # U-19Pu-10Zr has rho = 2e13 (reference value)
    reduction_factor = 1.0 - 0.04 * pu_content
    
    # Ensure minimum reasonable value
    return max(rho_base * reduction_factor, 1.5e13)

def estimate_peak_temperature_pu(pu_content, zr_content=10):
    """
    Estimate peak swelling temperature based on Pu content.
    
    Higher Pu content increases peak temperature due to:
    - Modified fission gas production
    - Changed thermal properties
    """
    # Base: 700 K for U-10Zr
    # Each % Pu adds ~2.6 K (so 19% Pu adds ~50K → 750K)
    return 700 + 2.6 * pu_content

# Display parameter estimates
print("=" * 80)
print("U-PU-ZR ALLOY: PARAMETER VARIATION WITH PU CONTENT")
print("(Zr content fixed at 10 wt%)")
print("=" * 80)
print(f"{'Composition':<25} {'Pu (wt%)':<12} {'Dislocation (m^-2)':<25} {'Peak T (K)':<15}")
print("-" * 80)

pu_params = {}
for pu in pu_contents:
    rho = estimate_dislocation_density_pu(pu)
    T_peak = estimate_peak_temperature_pu(pu)
    comp_name = f'U-{pu}Pu-10Zr'
    pu_params[pu] = {'rho': rho, 'T_peak': T_peak, 'name': comp_name}
    print(f"{comp_name:<25} {pu:<12} {rho:.2e}            {T_peak:<15.0f}")

print("\n" + "=" * 80)
print("PHYSICAL INTERPRETATION:")
print("=" * 80)
print("• Higher Pu content → lower dislocation density → reduced swelling")
print("• Higher Pu content → higher peak temperature → shifts swelling to higher T")
print("• Pu content is a key design parameter for transmutation fuels")
print("=" * 80)

In [ ]:
# Run simulations for different Pu contents
print("Running simulations for U-Pu-Zr alloys with varying Pu content...")
print("(This may take 30-60 seconds...)\n")

pu_results = {}

for pu in pu_contents:
    print(f"Simulating U-{pu}Pu-10Zr...", end=" ")
    
    # Get estimated parameters
    rho = pu_params[pu]['rho']
    T = pu_params[pu]['T_peak']
    
    # Create parameter set
    params = create_default_parameters()
    params['dislocation_density'] = rho
    params['temperature'] = T
    params['fission_rate'] = 2e20
    
    # Recalculate diffusion at this temperature
    from gas_swelling.params.parameters import BOLTZMANN_CONSTANT_EV
    Dgb = (params['Dgb_prefactor'] * np.exp(-params['Dgb_activation_energy'] / (BOLTZMANN_CONSTANT_EV * T)) + 
          params['Dgb_fission_term'] * params['fission_rate'])
    params['Dgb'] = Dgb
    params['Dgf'] = Dgb * params['Dgf_multiplier']
    
    # Run simulation
    model = GasSwellingModel(params)
    result = model.solve(t_span=(0, sim_time_sec), t_eval=t_eval)
    
    # Calculate swelling
    Vb = (4/3) * np.pi * result['Rcb']**3 * result['Ccb']
    Vf = (4/3) * np.pi * result['Rcf']**3 * result['Ccf']
    swelling = (Vb + Vf) * 100
    
    pu_results[pu] = {
        'swelling': swelling,
        'Rcb': result['Rcb'],
        'Ccb': result['Ccb'],
        'T': T,
        'name': pu_params[pu]['name']
    }
    
    print(f"✓ T={T}K, Swelling={swelling[-1]:.4f}%")

print("\n✅ All Pu content simulations completed!")

In [ ]:
# Visualize Pu content effects
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Effect of Plutonium Content in U-Pu-Zr Alloys', fontsize=14, fontweight='bold')

# Plot 1: Swelling evolution
colors_pu = plt.cm.plasma(np.linspace(0.2, 0.9, len(pu_contents)))
for i, pu in enumerate(pu_contents):
    axes[0].plot(time_days, pu_results[pu]['swelling'],
               linewidth=2, color=colors_pu[i], label=f'{pu}% Pu')
axes[0].set_xlabel('Time (days)', fontsize=11)
axes[0].set_ylabel('Swelling (%)', fontsize=11)
axes[0].set_title('Swelling Evolution', fontweight='bold')
axes[0].legend()

# Plot 2: Final swelling vs Pu content
final_swelling_pu = [pu_results[pu]['swelling'][-1] for pu in pu_contents]
axes[1].plot(pu_contents, final_swelling_pu, 'o-', linewidth=2, markersize=10, color='darkred')
axes[1].set_xlabel('Pu Content (wt%)', fontsize=11)
axes[1].set_ylabel('Final Swelling (%)', fontsize=11)
axes[1].set_title('Pu Content Effect', fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Add value labels
for i, (pu, swell) in enumerate(zip(pu_contents, final_swelling_pu)):
    axes[1].text(pu, swell, f'{swell:.3f}%', ha='center', va='bottom', fontsize=9)

# Plot 3: Peak temperature vs Pu content
peak_temps_pu = [pu_results[pu]['T'] for pu in pu_contents]
axes[2].plot(pu_contents, peak_temps_pu, 's-', linewidth=2, markersize=10, color='purple')
axes[2].set_xlabel('Pu Content (wt%)', fontsize=11)
axes[2].set_ylabel('Peak Temperature (K)', fontsize=11)
axes[2].set_title('Peak Temperature Shift', fontweight='bold')
axes[2].grid(True, alpha=0.3)

# Add value labels
for i, (pu, T) in enumerate(zip(pu_contents, peak_temps_pu)):
    axes[2].text(pu, T+2, f'{T}K', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\n📊 SUMMARY: PU CONTENT EFFECTS")
print("="*60)
print(f"Swelling reduction: {(final_swelling_pu[0]/final_swelling_pu[-1]-1)*100:.1f}% from 10% to 25% Pu")
print(f"Peak temperature shift: {max(peak_temps_pu) - min(peak_temps_pu)}K higher")
print("\nKey Insight: Higher Pu content reduces swelling but requires higher")
print("operating temperatures to achieve peak swelling behavior.")

## 5. Creating Custom Material Compositions

Now let's create helper functions to easily define custom material compositions for your research.

In [ ]:
def create_custom_uzr_composition(zr_content, temperature=None, fission_rate=2e20):
    """
    Create a custom U-Zr alloy parameter set.
    
    Parameters
    ----------
    zr_content : float
        Zirconium content in weight percent (typically 6-12%)
    temperature : float, optional
        Operating temperature in Kelvin. If None, uses estimated peak temperature.
    fission_rate : float, optional
        Fission rate in fissions/(m^3·s). Default is 2e20.
    
    Returns
    -------
    params : dict
        Complete parameter dictionary for the custom composition.
    
    Examples
    --------
    >>> params = create_custom_uzr_composition(zr_content=10, temperature=700)
    >>> params = create_custom_uzr_composition(zr_content=8)  # Auto-select peak T
    """
    # Estimate material properties based on composition
    rho = estimate_dislocation_density_zr(zr_content)
    T_peak = estimate_peak_temperature_zr(zr_content)
    
    # Use provided temperature or estimated peak
    T = temperature if temperature is not None else T_peak
    
    # Create parameters
    params = create_default_parameters()
    params['dislocation_density'] = rho
    params['temperature'] = T
    params['fission_rate'] = fission_rate
    params['Fnb'] = 1e-5  # Standard bulk nucleation for U-Zr
    params['Fnf'] = 1e-5  # Standard boundary nucleation for U-Zr
    
    # Recalculate diffusion at this temperature
    from gas_swelling.params.parameters import BOLTZMANN_CONSTANT_EV
    Dgb = (params['Dgb_prefactor'] * np.exp(-params['Dgb_activation_energy'] / (BOLTZMANN_CONSTANT_EV * T)) + 
          params['Dgb_fission_term'] * fission_rate)
    params['Dgb'] = Dgb
    params['Dgf'] = Dgb * params['Dgf_multiplier']
    
    return params

def create_custom_upuzr_composition(pu_content, zr_content=10, temperature=None, fission_rate=2e20):
    """
    Create a custom U-Pu-Zr alloy parameter set.
    
    Parameters
    ----------
    pu_content : float
        Plutonium content in weight percent (typically 10-30%)
    zr_content : float, optional
        Zirconium content in weight percent. Default is 10%.
    temperature : float, optional
        Operating temperature in Kelvin. If None, uses estimated peak temperature.
    fission_rate : float, optional
        Fission rate in fissions/(m^3·s). Default is 2e20.
    
    Returns
    -------
    params : dict
        Complete parameter dictionary for the custom composition.
    
    Examples
    --------
    >>> params = create_custom_upuzr_composition(pu_content=19, zr_content=10)
    >>> params = create_custom_upuzr_composition(pu_content=25, temperature=800)
    """
    # Estimate material properties based on composition
    rho = estimate_dislocation_density_pu(pu_content, zr_content)
    T_peak = estimate_peak_temperature_pu(pu_content, zr_content)
    
    # Use provided temperature or estimated peak
    T = temperature if temperature is not None else T_peak
    
    # Create parameters
    params = create_default_parameters()
    params['dislocation_density'] = rho
    params['temperature'] = T
    params['fission_rate'] = fission_rate
    params['Fnb'] = 1e-5  # Standard bulk nucleation for U-Pu-Zr
    params['Fnf'] = 1e-5  # Standard boundary nucleation for U-Pu-Zr
    
    # Recalculate diffusion at this temperature
    from gas_swelling.params.parameters import BOLTZMANN_CONSTANT_EV
    Dgb = (params['Dgb_prefactor'] * np.exp(-params['Dgb_activation_energy'] / (BOLTZMANN_CONSTANT_EV * T)) + 
          params['Dgb_fission_term'] * fission_rate)
    params['Dgb'] = Dgb
    params['Dgf'] = Dgb * params['Dgf_multiplier']
    
    return params

print("✅ Custom composition helper functions created!")
print("\nAvailable functions:")
print("  • create_custom_uzr_composition(zr_content, temperature, fission_rate)")
print("  • create_custom_upuzr_composition(pu_content, zr_content, temperature, fission_rate)")

In [ ]:
# Example: Create and compare custom compositions
print("=" * 80)
print("EXAMPLE: CREATING CUSTOM COMPOSITIONS")
print("=" * 80)

# Example 1: U-9Zr at 725 K
print("\nExample 1: U-9%Zr at 725 K")
params1 = create_custom_uzr_composition(zr_content=9, temperature=725)
print(f"  Dislocation density: {params1['dislocation_density']:.2e} m^-2")
print(f"  Temperature: {params1['temperature']} K")
print(f"  Fission rate: {params1['fission_rate']:.2e} fissions/(m^3·s)")

# Example 2: U-22Pu-8Zr at peak temperature
print("\nExample 2: U-22%Pu-8%Zr at peak temperature")
params2 = create_custom_upuzr_composition(pu_content=22, zr_content=8)
print(f"  Dislocation density: {params2['dislocation_density']:.2e} m^-2")
print(f"  Temperature (auto-selected peak): {params2['temperature']} K")
print(f"  Fission rate: {params2['fission_rate']:.2e} fissions/(m^3·s)")

# Example 3: High fission rate scenario
print("\nExample 3: U-10Zr at high fission rate (5e20)")
params3 = create_custom_uzr_composition(zr_content=10, fission_rate=5e20)
print(f"  Fission rate: {params3['fission_rate']:.2e} fissions/(m^3·s)")
print(f"  → 2.5x higher than standard (2e20)")

print("\n" + "=" * 80)
print("These custom parameter sets are ready for simulation!")
print("=" * 80)

## 6. Practical Example: Design Study for New Fuel Composition

Let's perform a practical design study: comparing a proposed new composition against standard fuels.

In [ ]:
# Design study: Compare U-12Pu-12Zr against standard fuels
print("=" * 80)
print("DESIGN STUDY: U-12%Pu-12%Zr PROPOSED COMPOSITION")
print("=" * 80)

# Create parameter sets for comparison
compositions = {
    'Standard U-10Zr': create_custom_uzr_composition(10, temperature=700),
    'Standard U-19Pu-10Zr': create_custom_upuzr_composition(19, 10),
    'Proposed U-12Pu-12Zr': create_custom_upuzr_composition(12, 12)
}

# Run simulations
print("\nRunning design study simulations...")
print("(This may take 30-60 seconds...)\n")

design_results = {}
for name, params in compositions.items():
    print(f"Simulating {name}...", end=" ")
    
    model = GasSwellingModel(params)
    result = model.solve(t_span=(0, sim_time_sec), t_eval=t_eval)
    
    Vb = (4/3) * np.pi * result['Rcb']**3 * result['Ccb']
    Vf = (4/3) * np.pi * result['Rcf']**3 * result['Ccf']
    swelling = (Vb + Vf) * 100
    
    design_results[name] = {
        'swelling': swelling,
        'Rcb': result['Rcb'],
        'Ccb': result['Ccb'],
        'T': params['temperature'],
        'rho': params['dislocation_density']
    }
    
    print(f"✓ Swelling={swelling[-1]:.4f}%")

print("\n✅ Design study completed!")

In [ ]:
# Visualize design study results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Design Study: Proposed U-12Pu-12Zr Composition', fontsize=14, fontweight='bold')

# Color scheme
colors_design = ['blue', 'green', 'orange']
names_design = list(design_results.keys())

# Plot 1: Swelling evolution
for i, name in enumerate(names_design):
    axes[0, 0].plot(time_days, design_results[name]['swelling'],
                   linewidth=2.5, color=colors_design[i], label=name)
axes[0, 0].set_xlabel('Time (days)', fontsize=11)
axes[0, 0].set_ylabel('Swelling (%)', fontsize=11)
axes[0, 0].set_title('Swelling Evolution', fontweight='bold')
axes[0, 0].legend()

# Plot 2: Final swelling comparison
final_swell_design = [design_results[name]['swelling'][-1] for name in names_design]
bars = axes[0, 1].bar(range(len(names_design)), final_swell_design,
                       color=colors_design, edgecolor='black', linewidth=1.5)
axes[0, 1].set_xticks(range(len(names_design)))
axes[0, 1].set_xticklabels(names_design, rotation=20, ha='right', fontsize=9)
axes[0, 1].set_ylabel('Final Swelling (%)', fontsize=11)
axes[0, 1].set_title('Final Swelling Comparison', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Add value labels
for i, bar in enumerate(bars):
    height = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 3: Dislocation density comparison
rho_design = [design_results[name]['rho'] for name in names_design]
axes[1, 0].bar(range(len(names_design)), np.array(rho_design)/1e13,
               color=colors_design, edgecolor='black', linewidth=1.5)
axes[1, 0].set_xticks(range(len(names_design)))
axes[1, 0].set_xticklabels(names_design, rotation=20, ha='right', fontsize=9)
axes[1, 0].set_ylabel('Dislocation Density (×10¹³ m⁻²)', fontsize=11)
axes[1, 0].set_title('Microstructure Comparison', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 4: Temperature comparison
T_design = [design_results[name]['T'] for name in names_design]
axes[1, 1].bar(range(len(names_design)), T_design,
               color=colors_design, edgecolor='black', linewidth=1.5)
axes[1, 1].set_xticks(range(len(names_design)))
axes[1, 1].set_xticklabels(names_design, rotation=20, ha='right', fontsize=9)
axes[1, 1].set_ylabel('Temperature (K)', fontsize=11)
axes[1, 1].set_title('Operating Temperature', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Analysis
print("\n" + "=" * 80)
print("DESIGN STUDY ANALYSIS")
print("=" * 80)

std_u10zr = final_swell_design[0]
std_upuzr = final_swell_design[1]
proposed = final_swell_design[2]

print(f"\n📊 Swelling Comparison:")
print(f"   Standard U-10Zr:       {std_u10zr:.4f}%")
print(f"   Standard U-19Pu-10Zr:  {std_upuzr:.4f}%")
print(f"   Proposed U-12Pu-12Zr:  {proposed:.4f}%")

print(f"\n🎯 Key Findings:")
print(f"   vs U-10Zr:     {(proposed/std_u10zr-1)*100:+.1f}% swelling change")
print(f"   vs U-19Pu-10Zr: {(proposed/std_upuzr-1)*100:+.1f}% swelling change")

print(f"\n🌡️  Temperature: {T_design[2]} K (vs {T_design[0]} K for U-10Zr)")
print(f"🔬 Microstructure: {rho_design[2]/1e13:.2f}×10¹³ m^-2 dislocation density")

print("\n" + "=" * 80)
print("💡 RECOMMENDATION:")
if proposed < std_u10zr and proposed < std_upuzr:
    print("✓ Proposed composition shows LOWER swelling than both standards!")
    print("  → Promising candidate for reduced swelling applications")
elif proposed < std_u10zr:
    print("✓ Proposed composition shows lower swelling than U-10Zr")
    print("  → Consider for applications requiring U-10Zr baseline with improvements")
else:
    print("⚠ Proposed composition has higher swelling than standards")
    print("  → Further optimization needed")
print("=" * 80)

## 7. Summary and Best Practices

### Key Takeaways

1. **Dislocation Density** is the primary parameter differentiating fuel compositions
   - U-Zr alloys: ~7×10¹³ m⁻²
   - U-Pu-Zr alloys: ~2×10¹³ m⁻²
   - High-purity U: ~1×10¹⁵ m⁻²

2. **Peak Swelling Temperature** varies with composition:
   - U-Zr: 700 K (at 10% Zr)
   - U-Pu-Zr: 750 K (at 19% Pu)
   - Pure U: 673 K

3. **Zr Content** effects:
   - Higher Zr → higher peak temperature
   - Moderate effect on dislocation density

4. **Pu Content** effects:
   - Higher Pu → lower dislocation density → lower swelling
   - Higher Pu → higher peak temperature

### Best Practices for Custom Compositions

```python
# 1. Use helper functions for consistency
params = create_custom_uzr_composition(zr_content=10, temperature=700)

# 2. Always validate against experimental data
# Compare model output to experimental swelling data

# 3. Consider temperature dependence
# Run temperature sweeps to find peak swelling for your composition

# 4. Document your parameter choices
# Keep notes on why you chose specific values
```

### Parameter Selection Flowchart

1. **Start with composition** → U-Zr or U-Pu-Zr?
2. **Determine alloy content** → What % Zr? What % Pu?
3. **Estimate dislocation density** → Use empirical relationships
4. **Select temperature** → Use peak temperature or operating condition
5. **Validate** → Compare to experimental data if available
6. **Iterate** → Adjust parameters as needed

### Next Steps

- **Explore other notebooks**:
  - `02_Parameter_Sweep_Studies.ipynb`: Systematic parameter variations
  - `04_Experimental_Data_Comparison.ipynb`: Validation against experiments

- **Read the documentation**:
  - `docs/parameter_reference.md`: Complete parameter guide
  - `docs/tutorials/rate_theory_fundamentals.md`: Physics background

- **Run example scripts**:
  - `examples/temperature_sweep_plotting.py`: Temperature studies

---

**🎉 Congratulations!** You've completed the Custom Material Composition tutorial.

You now have the skills to:
- Create custom material parameter sets for U-Zr and U-Pu-Zr alloys
- Understand how composition affects swelling behavior
- Perform design studies for new fuel compositions
- Make informed decisions about parameter selection

Happy researching! 🚀